<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/05_lyapunov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lyapunov stability

### Lyapunov Refresher
- $V: \mathbb{R}^n \rightarrow \mathbb{R}$, a scalar valued function.
- W.L.O.G, let $x_\mathrm{eq}=0$ be an equilibrium point where $f(x_\mathrm{eq})=0$.
- $V$ positive definite, $\dot V \le 0$ near the equilibrium $\Rightarrow$ **Lyapunov stable**.
- $\dot V < 0$ near the equilibrium $\Rightarrow$ **locally asymptotically stable**.
- If these hold only on a subset of the state space, stability is **local** (not global).

In [ ]:
from typing import Callable
import jax.numpy as jnp
import matplotlib.pyplot as plt
import functools
import jax
import scipy.linalg


In [ ]:
### Helper functions

# simulate ODE dynamics using Euler's method, x_dot = f(x)
def simulate_ode_dynamics(
    ode_dynamics: Callable[[jnp.ndarray], jnp.ndarray],
    initial_state: float,
    tmax: float,
) -> jnp.ndarray:
    dt = 1e-2
    num_steps = int(tmax / dt)
    ts = jnp.linspace(0, tmax, num_steps)
    xs = [initial_state]
    for i in range(1, num_steps):
        xs.append(xs[-1] + ode_dynamics(xs[-1]) * dt)
    x = jnp.array(xs)
    return x, ts

# simulate ODE dynamics using Euler's method, x_dot = f(x, u)
def simulate_ode_dynamics_control(
    ode_dynamics: Callable[[jnp.ndarray, jnp.ndarray], jnp.ndarray],
    initial_state: float,
    controls: jnp.ndarray,
    dt=1e-2,
) -> jnp.ndarray:
    num_steps = controls.shape[0]
    ts = [0.0]
    xs = [initial_state]
    for i in range(num_steps):
        xs.append(xs[-1] + ode_dynamics(xs[-1], controls[i]) * dt)
        ts.append(ts[-1] + dt)
    x = jnp.array(xs)
    ts = jnp.array(ts)
    return x, ts


## 1. Getting familiar with Lyapunov functions

Let's continue analyzing the system studied from Exercise 4. Let's consider the following 1D ODE dynamics, $\dot{x} = \alpha x + \beta x^3$, $\alpha, \beta \in \mathbb{R}$.

Let us consider a candidate Lyapunov function $V(x) = \frac{1}{2}x^2$.



In [ ]:
def cubic_ode(x: jnp.ndarray, alpha: float, beta: float) -> jnp.ndarray:
    return alpha * x + beta * x**3

def lyapunov_function(x: jnp.ndarray) -> float:
    '''x can be scalar or a vector'''
    return 0.5 * jnp.dot(x, x)


### (i) For a given set of dynamics, and a lyapunov function, implement a function that computes $\dot{V}$.

Recall that $\dot{V} = \nabla V(x)^T f(x)$ where $\dot{x} = f(x)$.

In [ ]:
def lyapunov_function_dot(x: jnp.ndarray, dynamics: Callable[[jnp.ndarray], jnp.ndarray],
                          lyapunov_function: Callable[[jnp.ndarray], float]) -> float:
    """Compute the time derivative of the Lyapunov function V at state x.

    Recall that \\dot{V} = \nabla V(x)^T f(x) where \\dot{x} = f(x).

    Args:
        x: The current state.
        dynamics: A function that takes in the state and returns the time derivative of the state.
        V: The Lyapunov function.

    Returns:
        The time derivative of the Lyapunov function at state x.
    """
    # Compute the gradient of V at x
    grad_V = jax.grad(lyapunov_function)(x)
    # Compute the dynamics at x
    f_x = dynamics(x)
    # Compute the time derivative of V
    V_dot = jnp.dot(grad_V, f_x)
    return V_dot

### (ii) Let $\alpha = -1, \beta=1$. Verify your implementation above.
Verify that your hand calculation matches the output for your function.

In [ ]:
# define the ODE dynamics with specific alpha and beta values
dynamics_ode = functools.partial(cubic_ode, alpha=-1.0, beta=1.0)

[in-class discussion, student response]

In [ ]:
# Hand calculation for lyapunov_function_dot with alpha = -1, beta = 1
def lyapunov_function_dot_analytic(x):
    return -x**2 + x**4


In [ ]:
xs = jnp.arange(-2.0, 2.0, 0.01)
hand_calculation = jax.vmap(lyapunov_function_dot_analytic)(xs)
implementation = jax.vmap(functools.partial(lyapunov_function_dot, dynamics=dynamics_ode, lyapunov_function=lyapunov_function))(xs)

# check if your hand calculation matches the output for your function
print("Number of discrepancies: ", (~jnp.isclose(implementation, hand_calculation)).sum())

### (iii) Determine what values of $x$ results in $\dot{V}(x) < 0$.

Use your analytic expression to determine the set of $x$ values. Then verify your answer by plotting it out.
Going back to what you learned abotu equilibria from Exercise 4, does your answer make sense? Briefly explain why.


In [ ]:
# Please run this code AFTER, to help verify your answer.
xs = jnp.arange(-2.0, 2.0, 0.01)
lyapunov_dots = jax.vmap(functools.partial(lyapunov_function_dot, dynamics=dynamics_ode, lyapunov_function=lyapunov_function))(xs)
plt.plot(xs, lyapunov_dots)
plt.fill_between(xs, lyapunov_dots, where=(lyapunov_dots < 0), color='tab:blue', alpha=0.3, label=r'$\dot{V}(x) < 0$')
plt.grid(alpha=0.5)
plt.ylim(-.5, .5)


### (iv) Pick some initial states within that region from (iii), and simulate the trajectory and lyapunov value

In [ ]:
x_lower = -4.0 # update me
x_upper = 4.0 # update me
tmax = 10.0
initial_xs = jnp.linspace(x_lower, x_upper, 11)
trajectories, ts = jax.vmap(simulate_ode_dynamics, in_axes=(None, 0, None))(dynamics_ode, initial_xs, tmax)

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
plt.plot(ts[0], trajectories.T, alpha=0.5)
plt.grid(alpha=0.5, zorder=-5)
plt.xlabel('time')
plt.ylabel('x(t)')
plt.title('State trajectories')

plt.subplot(1,2,2)
lyapunov_values = jax.vmap(jax.vmap(lyapunov_function))(trajectories)
plt.plot(ts[0], lyapunov_values.T, alpha=0.5)
plt.grid(alpha=0.5, zorder=-5)
plt.xlabel('time')
plt.ylabel('Lyapunov value V(x)')
plt.title('Lyapunov values')

### (v) Consider the case where $\beta = 0.25, 0.81, 4, 9$. Deduce which values of $x$ results in $\dot{V} < 0$.

[student response here]

In [ ]:
# Please run this code AFTER, to help verify your answer.
xs = jnp.arange(-2.0, 2.0, 0.01)
betas = [0.25, 0.81, 4, 9]
for beta in betas:
    dynamics_ode_beta = functools.partial(cubic_ode, alpha=-1.0, beta=beta)
    lyapunov_dots = jax.vmap(functools.partial(lyapunov_function_dot, dynamics=dynamics_ode_beta, lyapunov_function=lyapunov_function))(xs)
    plt.plot(xs, lyapunov_dots, label=f'beta={beta}')
    plt.fill_between(xs, lyapunov_dots, where=(lyapunov_dots < 0), alpha=0.3)
plt.ylim(-1.5, 0.5)
plt.grid()
plt.legend()
plt.xlabel('x')
plt.ylabel(r'$\dot{V}(x)$')

### (vi) For $\alpha = -1, \beta > 0$, is $x$ **globally** asymptotically stable under the given $V$? Why or why not?

[student response here]

## 2. Linear state feedback

We consider the continuous-time LTI system
\begin{aligned}
\dot x = A x + B u,\qquad x\in\mathbb{R}^2,\ u\in\mathbb{R},
\end{aligned}

with

\begin{aligned}
A=\begin{bmatrix}0 & 1\\[3pt] 1 & 0\end{bmatrix},\quad B=\begin{bmatrix}0\\[3pt] 1\end{bmatrix}.
\end{aligned}

This $A$ has eigenvalues $\{+1,-1\}$ $\Rightarrow$ **unstable** (one positive).


In [ ]:
def dynamics_ode(state: jnp.ndarray, control: jnp.ndarray) -> jnp.ndarray:
    A = jnp.array([[0.0, 1.0],
                   [1.0, 0.0]])
    B = jnp.array([[0.],
                   [1.0]])
    return A @ state + B @ control

# example usage
state = jnp.array([1.0, 2.0])
control = jnp.array([0.5])
x_dot = dynamics_ode(state, control)

### (i) Understand the open-loop instability of the system

The open-loop dynamics of the system is when $u\equiv 0$, i.e., $\dot x = A x$.

Find the eigenvectors corresponding to each of the eigenvalues.


$\lambda = 1$, $v = \ldots$

$\lambda = -1$, $v = \ldots$

Below, we simulate initial states $ x_0 \in \{(1,0)^\top,\ (0,1)^\top,\ (-1,1)^\top,\ (1,-1)^\top\}. $ on $t\in[0, 2]$ with $ \Delta t=0.02 $ and:

1) Plot **phase-plane** trajectories (overlay all ICs).

3) Plot $\|x(t)\|$ vs $t$ (semi-log or linear) to show growth.

Which trajectories diverge? Along which directions is growth fastest? How does this relate to the eigenvector for the unstable eigenvalue $\lambda=+1$?

In [ ]:
dt = 0.02
tmax = 2.0
n_steps = int(tmax / dt)
initial_states = jnp.array([[1.0, 0.0],
                            [0.0, 1.0],
                            [-1.0, 1.0],
                            [1.0, -1.0]])
control = jnp.zeros([n_steps, 1])  # zero control input
xs, ts = jax.vmap(simulate_ode_dynamics_control, [None, 0, None, None])(dynamics_ode, initial_states, control, dt)

N = 21
X, Y = jnp.meshgrid(jnp.linspace(-3, 3, N), jnp.linspace(-3, 3, N))
XYs = jnp.stack([X, Y], axis=-1).reshape(-1, 2)  # shape (400, 2)
x_dot = jax.vmap(dynamics_ode, [0, None])(XYs, jnp.zeros([1])).reshape(N, N, 2)


plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(xs[:, :, 0].T, xs[:, :, 1].T, alpha=0.5)
plt.scatter(xs[:, :1, 0].T, xs[:, :1, 1].T, alpha=0.5, label='start')
plt.quiver(X, Y, x_dot[..., 0], x_dot[..., 1], color='tab:blue')


plt.xlabel('x1')
plt.ylabel('x2')
plt.title('State trajectories')
plt.grid(alpha=0.5, zorder=-5)
plt.legend()

plt.subplot(1,2,2)
xnorm = jnp.linalg.norm(xs, axis=-1)
plt.plot(ts[0], xnorm.T, alpha=0.5)
plt.xlabel('Time')
plt.ylabel('State norm ||x||')
plt.title('State norm over Time')
plt.grid(alpha=0.5, zorder=-5)
plt.yscale('log')


### (ii) Design a state feedback controller  $u = -Kx$ using `pole placement`

We want to find a controller (or policy) that computes what $u$ should be given $x$ such that the system becomes stable.

To do this, we use full-state feedback: $u=-Kx$, $K=[k_1, k_2]\in\mathbb{R}^{1\times 2}$, resulting in $\dot x = (A - BK)x$.


Suppose that we wish to design $K$ such that the *closed-loop* poles are located at $-1$ and $-2$. This is equivalent to saying that the *eigenvalues* are $-1$ and $-2$.

What values of $K=[k_1\ k_2]$ should we pick such that $A_{\text{cl}}=A-BK$ has your desired poles?


[student response where]

In [ ]:
K = jnp.array([[3.0, 3.0]])  # update me
A = jnp.array([[0.0, 1.0],
                   [1.0, 0.0]])
B = jnp.array([[0.],
                   [1.0]])
A_cl = A - B @ K
eigenvalues, eigenvectors = jnp.linalg.eig(A_cl)
print("Closed-loop eigenvalues:", eigenvalues)


# closed-loop dynamics with state feedback u = -Kx
def dynamics_ode_closedloop(state: jnp.ndarray, K: jnp.ndarray) -> jnp.ndarray:
    A = jnp.array([[0.0, 1.0],
                   [1.0, 0.0]])
    B = jnp.array([[0.],
                   [1.0]])
    u = -K @ state  # state feedback control law
    return A @ state + B @ u



In [ ]:
# simulate closed-loop dynamics
dt = 0.02
tmax = 2.0
n_steps = int(tmax / dt)
initial_states = jnp.array([[1.0, 0.0],
                            [0.0, 1.0],
                            [-1.0, 1.0],
                            [1.0, -1.0]])
control = jnp.zeros([n_steps, 1])  # zero control input
closed_loop_dynamics = functools.partial(dynamics_ode_closedloop, K=K)
xs, ts = jax.vmap(simulate_ode_dynamics, [None, 0, None])(closed_loop_dynamics, initial_states, tmax)

N = 21
X, Y = jnp.meshgrid(jnp.linspace(-3, 3, N), jnp.linspace(-3, 3, N))
XYs = jnp.stack([X, Y], axis=-1).reshape(-1, 2)  # shape (400, 2)
x_dot = jax.vmap(closed_loop_dynamics, [0])(XYs).reshape(N, N, 2)


plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(xs[:, :, 0].T, xs[:, :, 1].T, alpha=0.5)
plt.scatter(xs[:, :1, 0].T, xs[:, :1, 1].T, alpha=0.5, label='start')
plt.quiver(X, Y, x_dot[..., 0], x_dot[..., 1], color='tab:blue')


plt.xlabel('x1')
plt.ylabel('x2')
plt.title('State trajectories')
plt.grid(alpha=0.5, zorder=-5)
plt.legend()

plt.subplot(1,2,2)
xnorm = jnp.linalg.norm(xs, axis=-1)
plt.plot(ts[0], xnorm.T, alpha=0.5)
plt.xlabel('Time')
plt.ylabel('State norm ||x||')
plt.title('State norm over Time')
plt.grid(alpha=0.5, zorder=-5)
plt.yscale('log')
plt.tight_layout()


Yay! You just designed a controller that stablizes a naturally unstable system!
Now we want to prove that this system is indeed truly stable. In other words, we would like to find a *certificate* that proves that the closed-loop system is stable.

## 3. Lyapunov certificate via the Lyapunov equation


**Theorem** (continuous time version). Given any $Q>0$, there exists a unique $P>0$ satisfying $A^TP + PA + Q = 0$ if and only if the linear system $\dot {x}=Ax$ is globally asymptotically stable. The quadratic function $V(x)=x^{T}Px$ is a Lyapunov function that can be used to verify stability.

### (i) Let's find $P$ using the Lyapunov equation.

First, we need $Q\succ 0$. To keep things simple, let us use $Q=I$.

Read up on `scipy.linalg.solve_continuous_lyapunov` and understand it's expected inputs and usage.


In [ ]:
?scipy.linalg.solve_continuous_lyapunov

In [ ]:
Q = jnp.eye(2)  # update me
P = scipy.linalg.solve_continuous_lyapunov(A_cl.T, -Q)

print(P)

### (ii) Let's plot the level sets of $x^TPx$.

In [ ]:
N = 31
X, Y = jnp.meshgrid(jnp.linspace(-3.0, 3.0, N), jnp.linspace(-3.0, 3.0, N))
XYs = jnp.stack([X, Y], axis=-1).reshape(-1, 2)  # shape (961, 2)
V_values = jax.vmap(lambda x: x.T @ P @ x)(XYs).reshape(N, N)

plt.figure(figsize=(16,7))
plt.jet()
plt.subplot(1,2,1)
plt.contourf(X, Y, V_values, levels=20)
plt.colorbar()

# simulate closed-loop dynamics
dt = 0.02
tmax = 5.0
n_steps = int(tmax / dt)
initial_states = jax.random.uniform(jax.random.PRNGKey(1), (20, 2), minval=-2.0, maxval=2.0)
control = jnp.zeros([n_steps, 1])  # zero control input
closed_loop_dynamics = functools.partial(dynamics_ode_closedloop, K=K)
xs, ts = jax.vmap(simulate_ode_dynamics, [None, 0, None])(closed_loop_dynamics, initial_states, tmax)

plt.plot(xs[:, :, 0].T, xs[:, :, 1].T,)
plt.scatter(xs[:, :1, 0].T, xs[:, :1, 1].T, label='start')

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('State trajectories')
plt.grid(alpha=0.5, zorder=-5)
plt.legend()
plt.axis('equal')
plt.xlim(-3, 3)
plt.ylim(-3, 3)

plt.subplot(1,2,2)

Vs = jax.vmap(jax.vmap(lambda x: x.T @ P @ x))(xs)
plt.plot(ts[0], Vs.T, alpha=0.5)
plt.grid()
plt.xlabel('Time')
plt.ylabel('Lyapunov value V(x)')
plt.title('Lyapunov values over Time')